# Hospital Operations & Revenue Risk Intelligence Platform
## Phase 1 — SQL Analytics Layer (Business Intelligence Foundation)

**Programme:** Advanced Certificate Programme in Applied AI and Machine Learning
**Platform:** Kaggle Notebook

---

### What This Phase Does

This is **Phase 1** of the capstone. We build the data foundation for the entire system.

**The business problem:**
A multi-specialty hospital network is losing revenue because of insurance claim rejections, delayed payments, and inefficient patient flow across departments.

**What we build here:**
1. Load the 3 raw CSV files from the Kaggle dataset into a **SQLite relational database** (runs entirely inside Python — no separate software needed)
2. Enforce data integrity: primary keys, foreign keys, data type constraints, and indexes
3. Run **15 SQL queries** answering real operational and financial business questions
4. Run **5 data quality checks** to identify dirty or inconsistent records
5. Create **2 reusable SQL Views** (virtual tables) for convenience
6. Export **`master_visit_view.csv`** to `/kaggle/working/` — this flat file is the input for every phase after this

**The three data tables:**

| Table | Source File | What It Stores |
|---|---|---|
| `patients` | patients.csv | Patient demographics and insurance information |
| `visits` | visits.csv | Hospital visit records per patient |
| `billing` | billing.csv | Insurance billing and claim outcomes |

**Table relationships (how they connect):**
```
patients (patient_id PRIMARY KEY)
    |
    └──> visits (visit_id PRIMARY KEY, patient_id FOREIGN KEY)
              |
              └──> billing (bill_id PRIMARY KEY, visit_id FOREIGN KEY)
```
---


## Step 1 — Import Libraries

We need only two libraries for this phase:

- `sqlite3` — Python's **built-in** SQL database engine. No installation needed. Creates a full relational database that lives inside Python's memory.
- `pandas` — reads CSV files and displays SQL query results as clean, readable tables.

`os` is used to handle file paths cleanly across different environments.


In [1]:
import sqlite3       # Built-in Python SQL engine — no pip install needed
import pandas as pd  # For reading CSVs and displaying query results as tables
import os            # For building file paths

# Display settings — show all columns, and format decimal numbers cleanly
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")


Libraries imported successfully.


## Step 2 — Define File Paths

On Kaggle, every dataset you attach to a notebook is available at:
`/kaggle/input/<dataset-slug>/`

Our dataset slug is `healthcare_capstone_project`, so our three CSV files are at the paths defined below.

`/kaggle/working/` is where we save outputs. Files saved here persist in the session and can be downloaded. We will save `master_visit_view.csv` here at the end of this phase — Phase 2 will read it from the same location.


In [2]:
# ── Input paths: raw CSV files from the attached Kaggle dataset ──────────────
INPUT_DIR  = "/kaggle/input/healthcare_capstone_project"

PATIENTS_CSV = "/kaggle/input/datasets/yashrajrajarishi/healthcare-capstone-project/patients.csv"
VISITS_CSV   = "/kaggle/input/datasets/yashrajrajarishi/healthcare-capstone-project/visits.csv"
BILLING_CSV  = "/kaggle/input/datasets/yashrajrajarishi/healthcare-capstone-project/billing.csv"

# ── Output path: where we save files for Phase 2 and beyond ──────────────────
OUTPUT_DIR = "/kaggle/working"
MASTER_CSV = os.path.join(OUTPUT_DIR, "master_visit_view.csv")

# Confirm the input files exist before we proceed
for path in [PATIENTS_CSV, VISITS_CSV, BILLING_CSV]:
    status = "FOUND" if os.path.exists(path) else "MISSING — check dataset attachment"
    print(f"{status} : {path}")


FOUND : /kaggle/input/datasets/yashrajrajarishi/healthcare-capstone-project/patients.csv
FOUND : /kaggle/input/datasets/yashrajrajarishi/healthcare-capstone-project/visits.csv
FOUND : /kaggle/input/datasets/yashrajrajarishi/healthcare-capstone-project/billing.csv


## Step 3 — Load CSV Files into Python

`pd.read_csv()` reads a CSV file from disk and loads it into a **DataFrame**.
Think of a DataFrame as an in-memory spreadsheet — rows and columns, but fully programmable.

We load all three files and print their dimensions to confirm everything loaded correctly.


In [3]:
# Load all three raw CSV files into pandas DataFrames
patients_df = pd.read_csv(PATIENTS_CSV)
visits_df   = pd.read_csv(VISITS_CSV)
billing_df  = pd.read_csv(BILLING_CSV)

# Print dimensions: (rows, columns)
print(f"patients : {patients_df.shape[0]:,} rows  x  {patients_df.shape[1]} columns")
print(f"visits   : {visits_df.shape[0]:,} rows  x  {visits_df.shape[1]} columns")
print(f"billing  : {billing_df.shape[0]:,} rows  x  {billing_df.shape[1]} columns")


patients : 5,000 rows  x  7 columns
visits   : 25,000 rows  x  8 columns
billing  : 25,000 rows  x  7 columns


In [4]:
# Preview the first 5 rows of each table
# This confirms column names, data types, and sample values are as expected

print("PATIENTS — first 5 rows:")
display(patients_df.head())

print("\nVISITS — first 5 rows:")
display(visits_df.head())

print("\nBILLING — first 5 rows:")
display(billing_df.head())


PATIENTS — first 5 rows:


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date
0,1,53,M,Hyderabad,SecureLife,0,2025-05-14
1,2,42,M,Pune,HealthPlus,0,2025-11-18
2,3,56,F,Hyderabad,HealthPlus,0,2025-05-11
3,4,72,M,Chennai,HealthPlus,0,2025-12-18
4,5,40,F,Bangalore,CareOne,1,2025-01-23



VISITS — first 5 rows:


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118



BILLING — first 5 rows:


,bill_id,visit_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,1,"23,577.37",0.00,Rejected,16.00,2025-06-18
1,2,2,"38,178.81","38,178.81",Paid,18.00,2025-10-09
2,3,3,"5,038.97","5,038.97",Paid,NaN,2025-01-20
3,4,4,"22,813.34","22,813.34",Paid,16.00,2025-12-24
4,5,5,"27,106.95","27,106.95",Paid,14.00,2025-09-23


## Step 4 — Create the SQLite Relational Database

### What is SQLite?
SQLite is a lightweight, file-based SQL database. Here we use `":memory:"` which means the entire database lives in RAM — fast, and requires no setup or server.

### Why a relational database instead of just pandas?
The capstone requirement is to build an analytics layer on top of SQL. SQL is the industry standard for hospital data warehouses, EHR systems, and billing platforms. Designing queries here mirrors exactly how real hospital BI teams work.

### Key SQL concepts used in this step:

**Data Types** — what kind of value each column holds:
- `INTEGER` — whole numbers (1, 42, 5000)
- `TEXT` — any string ("Cardiology", "M", "Paid")
- `REAL` — decimal numbers (3.14, 25000.50)

**Constraints** — rules enforced on every row inserted:
- `PRIMARY KEY` — value must be unique; no two rows can have the same value; cannot be NULL
- `FOREIGN KEY` — value must exist in the referenced table; prevents orphaned records
- `NOT NULL` — column must always have a value
- `CHECK(condition)` — value must satisfy the condition (e.g., age must be between 0 and 130)

**Indexes** — pre-sorted lookup structures on specific columns:
- Without an index: every query scans ALL rows (slow on large tables)
- With an index: the database jumps directly to matching rows (fast)
- We index every column used in `WHERE` filters, `JOIN` conditions, or `GROUP BY`


In [5]:
# Create an in-memory SQLite database
# ":memory:" = database lives in RAM; fast and portable for Kaggle notebooks
# If you want to save to disk instead, replace ":memory:" with "hospital.db"

conn   = sqlite3.connect(":memory:")
cursor = conn.cursor()

print("SQLite database created in memory.")
print("Connection established.")


SQLite database created in memory.
Connection established.


In [6]:
# ── CREATE TABLES ────────────────────────────────────────────────────────────
# We drop existing tables first so this cell can be safely re-run without errors.
# Tables must be dropped in reverse dependency order:
# billing depends on visits, visits depends on patients — so drop billing first.

cursor.executescript(
"""
DROP TABLE IF EXISTS billing;
DROP TABLE IF EXISTS visits;
DROP TABLE IF EXISTS patients;

CREATE TABLE patients (
    patient_id         INTEGER  PRIMARY KEY,
    age                INTEGER  NOT NULL  CHECK(age >= 0 AND age <= 130),
    gender             TEXT     NOT NULL  CHECK(gender IN ('M', 'F')),
    city               TEXT     NOT NULL,
    insurance_provider TEXT     NOT NULL,
    chronic_flag       INTEGER  NOT NULL  CHECK(chronic_flag IN (0, 1)),
    registration_date  TEXT     NOT NULL
);

CREATE TABLE visits (
    visit_id             INTEGER  PRIMARY KEY,
    patient_id           INTEGER  NOT NULL,
    visit_date           TEXT     NOT NULL,
    department           TEXT     NOT NULL,
    visit_type           TEXT     NOT NULL  CHECK(visit_type IN ('OPD', 'ER', 'ICU')),
    length_of_stay_hours REAL              CHECK(length_of_stay_hours >= 0),
    risk_score           TEXT     NOT NULL  CHECK(risk_score IN ('Low', 'Medium', 'High')),
    doctor_id            INTEGER  NOT NULL,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
);

CREATE TABLE billing (
    bill_id         INTEGER  PRIMARY KEY,
    visit_id        INTEGER  NOT NULL,
    billed_amount   REAL     NOT NULL  CHECK(billed_amount >= 0),
    approved_amount REAL               CHECK(approved_amount >= 0),
    claim_status    TEXT     NOT NULL  CHECK(claim_status IN ('Paid', 'Pending', 'Rejected')),
    payment_days    REAL               CHECK(payment_days >= 0),
    billing_date    TEXT     NOT NULL,
    FOREIGN KEY (visit_id) REFERENCES visits(visit_id)
);
"""
)

print("Tables created:")
print("  patients — PRIMARY KEY: patient_id")
print("  visits   — PRIMARY KEY: visit_id | FOREIGN KEY: patient_id -> patients")
print("  billing  — PRIMARY KEY: bill_id  | FOREIGN KEY: visit_id  -> visits")


Tables created:
  patients — PRIMARY KEY: patient_id
  visits   — PRIMARY KEY: visit_id | FOREIGN KEY: patient_id -> patients
  billing  — PRIMARY KEY: bill_id  | FOREIGN KEY: visit_id  -> visits


In [7]:
# ── CREATE INDEXES ───────────────────────────────────────────────────────────
# An index is like a book's index — it pre-sorts a column so queries that
# filter or join on that column don't have to scan every row.
#
# Rule: index any column used in WHERE, JOIN ON, or GROUP BY.

cursor.executescript(
"""
CREATE INDEX IF NOT EXISTS idx_visits_date      ON visits(visit_date);
CREATE INDEX IF NOT EXISTS idx_visits_dept      ON visits(department);
CREATE INDEX IF NOT EXISTS idx_visits_risk      ON visits(risk_score);
CREATE INDEX IF NOT EXISTS idx_visits_patient   ON visits(patient_id);
CREATE INDEX IF NOT EXISTS idx_patients_insurer ON patients(insurance_provider);
CREATE INDEX IF NOT EXISTS idx_billing_status   ON billing(claim_status);
CREATE INDEX IF NOT EXISTS idx_billing_visit    ON billing(visit_id);
"""
)

print("Indexes created:")
print("  visits   : visit_date, department, risk_score, patient_id")
print("  patients : insurance_provider")
print("  billing  : claim_status, visit_id")


Indexes created:
  visits   : visit_date, department, risk_score, patient_id
  patients : insurance_provider
  billing  : claim_status, visit_id


### Index Documentation — Which Queries Benefit from Which Indexes

The table below maps each index to the specific queries it speeds up and explains why.

| Index | Column Indexed | Queries That Benefit | Why It Helps |
|---|---|---|---|
| `idx_visits_date` | `visits.visit_date` | Phase 2 time-based train/test split, Phase 6 drift detection | Avoids full table scan when filtering or ordering by date |
| `idx_visits_dept` | `visits.department` | A1 (top departments by volume), A2 (avg stay by dept), A3 (high-risk % by dept), B4 (realization ratio by dept) | Every department-level GROUP BY and WHERE filter uses this |
| `idx_visits_risk` | `visits.risk_score` | A3 (high-risk % per dept), A5 (doctors with most high-risk visits), C4 (null/invalid checks) | Speeds up all WHERE risk_score = 'High' filters |
| `idx_visits_patient` | `visits.patient_id` | A4 (avg visits per patient by city) — the inner subquery joins visits to patients on this column | Speeds up the JOIN between visits and patients |
| `idx_patients_insurer` | `patients.insurance_provider` | B1 (top insurers by billed amount), B2 (rejection rate by insurer), B3 (payment delay by insurer) | Every insurer-level GROUP BY traverses this join path |
| `idx_billing_status` | `billing.claim_status` | B2 (rejection rate), B3 (avg payment delay — filters WHERE claim_status = 'Paid'), C1 (unbilled visits) | Speeds up all WHERE and GROUP BY on claim_status |
| `idx_billing_visit` | `billing.visit_id` | C1 (visits without billing — LEFT JOIN on visit_id), C2 (ghost bills — LEFT JOIN on visit_id), B4, B5 | Every JOIN between billing and visits uses this column |

**General rule applied:** Any column appearing in a `WHERE`, `JOIN ON`, or `GROUP BY` clause in a frequently-run query was indexed. Columns used only in `SELECT` or `ORDER BY` were not indexed separately, as the performance gain there is marginal compared to filter and join columns.


## Step 5 — Load Data into the SQL Tables

`df.to_sql()` pushes all rows from a pandas DataFrame into the SQL table we just created.

- `if_exists='append'` — adds rows to the existing table without touching its schema or constraints
- `index=False` — do not write the pandas row-number index as a column in SQL


In [8]:
# Push all three DataFrames into the SQL tables
patients_df.to_sql("patients", conn, if_exists="append", index=False)
visits_df.to_sql(  "visits",   conn, if_exists="append", index=False)
billing_df.to_sql( "billing",  conn, if_exists="append", index=False)

# Verify row counts match the original CSV files exactly
r_p = pd.read_sql("SELECT COUNT(*) AS n FROM patients", conn).iloc[0, 0]
r_v = pd.read_sql("SELECT COUNT(*) AS n FROM visits",   conn).iloc[0, 0]
r_b = pd.read_sql("SELECT COUNT(*) AS n FROM billing",  conn).iloc[0, 0]

print(f"patients : {r_p:,} rows loaded")
print(f"visits   : {r_v:,} rows loaded")
print(f"billing  : {r_b:,} rows loaded")


patients : 5,000 rows loaded
visits   : 25,000 rows loaded
billing  : 25,000 rows loaded


---
## Section A — Operational Analysis

These queries answer **clinical operations** questions for hospital administrators and department heads.

The goal is to understand:
- Which departments are under the most load?
- Where are patients staying the longest?
- Where are the high-risk patients concentrated?
- Which doctors are managing the most complex cases?


### Query A1 — Top 10 Departments by Total Visit Volume

**Business question:** Which departments handle the most patient visits?
**Why it matters:** High-volume departments need proportionally more staff, beds, and equipment.

**SQL concepts in this query:**
- `COUNT(*)` — counts all rows in each group (= number of visits per department)
- `GROUP BY department` — collapses all rows with the same department into one result row before counting
- `ORDER BY total_visits DESC` — sorts results highest-to-lowest (`DESC` = descending)
- `LIMIT 10` — returns only the top 10 rows


In [9]:
query_a1 = """
    SELECT
        department,
        COUNT(*) AS total_visits
    FROM visits
    GROUP BY department
    ORDER BY total_visits DESC
    LIMIT 10
"""

df_a1 = pd.read_sql(query_a1, conn)
print("A1 — Top 10 Departments by Visit Volume:")
display(df_a1)

top = df_a1.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {top['department']} is the busiest department with {top['total_visits']:,} visits.")
print("  These high-volume departments should be prioritised for staffing and capacity planning.")


A1 — Top 10 Departments by Visit Volume:


,department,total_visits
0,General,4228
1,ER,4220
2,Neurology,4165
3,Orthopedics,4164
4,Cardiology,4159
5,ICU,4064



Business Insight:
  General is the busiest department with 4,228 visits.
  These high-volume departments should be prioritised for staffing and capacity planning.


### Query A2 — Top 5 Departments by Average Length of Stay

**Business question:** Which departments keep patients the longest on average?
**Why it matters:** Longer stays consume more nursing hours, beds, and consumables. These departments drive the highest per-visit cost.

**SQL concepts:**
- `AVG(column)` — calculates the arithmetic mean across all rows in each group
- `ROUND(value, 2)` — rounds the result to 2 decimal places for cleaner display
- `WHERE length_of_stay_hours IS NOT NULL` — excludes any rows where this value is missing before computing the average


In [10]:
query_a2 = """
    SELECT
        department,
        ROUND(AVG(length_of_stay_hours), 2) AS avg_length_of_stay_hours
    FROM visits
    WHERE length_of_stay_hours IS NOT NULL
    GROUP BY department
    ORDER BY avg_length_of_stay_hours DESC
    LIMIT 5
"""

df_a2 = pd.read_sql(query_a2, conn)
print("A2 — Top 5 Departments by Average Length of Stay (hours):")
display(df_a2)

top = df_a2.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {top['department']} averages {top['avg_length_of_stay_hours']:.1f} hours per visit — the longest.")
print("  Review discharge protocols and bed turnover rates in long-stay departments.")


A2 — Top 5 Departments by Average Length of Stay (hours):


,department,avg_length_of_stay_hours
0,Neurology,19.72
1,Orthopedics,19.66
2,Cardiology,19.60
3,ER,19.53
4,General,19.43



Business Insight:
  Neurology averages 19.7 hours per visit — the longest.
  Review discharge protocols and bed turnover rates in long-stay departments.


### Query A3 — Percentage of High-Risk Visits per Department

**Business question:** What fraction of each department's visits are classified as High Risk?
**Why it matters:** Departments with high proportions of high-risk visits need specialised clinical teams and rapid-response infrastructure.

**SQL concepts:**
- `CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END` — assigns 1 if the condition is true, 0 otherwise. This is SQL's version of an if-else statement.
- `SUM(CASE WHEN ...)` — adds up all the 1s, effectively counting only the rows where the condition was true
- `CAST(COUNT(*) AS REAL)` — converts the integer count to a decimal number so division gives a decimal result instead of rounding to 0
- `100.0 * ...` — multiplies by 100 to convert the ratio to a percentage


In [11]:
query_a3 = """
    SELECT
        department,
        COUNT(*) AS total_visits,
        SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) AS high_risk_visits,
        ROUND(
            100.0 * SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END)
            / CAST(COUNT(*) AS REAL),
            2
        ) AS high_risk_pct
    FROM visits
    GROUP BY department
    ORDER BY high_risk_pct DESC
"""

df_a3 = pd.read_sql(query_a3, conn)
print("A3 — High-Risk Visit Percentage by Department:")
display(df_a3)

top = df_a3.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {top['department']} has the highest High-Risk rate: {top['high_risk_pct']:.1f}% of all its visits.")
print("  Deploy senior clinicians and rapid-response teams to these departments.")


A3 — High-Risk Visit Percentage by Department:


,department,total_visits,high_risk_visits,high_risk_pct
0,ICU,4064,845,20.79
1,ER,4220,872,20.66
2,Neurology,4165,846,20.31
3,Orthopedics,4164,842,20.22
4,General,4228,839,19.84
5,Cardiology,4159,790,18.99



Business Insight:
  ICU has the highest High-Risk rate: 20.8% of all its visits.
  Deploy senior clinicians and rapid-response teams to these departments.


### Query A4 — Average Number of Visits per Patient by City

**Business question:** Which cities produce patients who visit the hospital most frequently?
**Why it matters:** High visit frequency could indicate a sicker population or strong hospital accessibility in that city — both are relevant for outreach and satellite clinic planning.

**SQL concepts:**
- **Subquery** — a query written inside another query. Here we first count visits per patient (inner query), then average those counts by city (outer query).
- `JOIN` — combines rows from two tables based on a matching column. `patients JOIN subquery ON patient_id` links each patient's city to their visit count.


In [12]:
query_a4 = """
    SELECT
        p.city,
        ROUND(AVG(vc.visits_per_patient), 2) AS avg_visits_per_patient
    FROM patients p
    JOIN (
        SELECT patient_id, COUNT(*) AS visits_per_patient
        FROM visits
        GROUP BY patient_id
    ) AS vc ON p.patient_id = vc.patient_id
    GROUP BY p.city
    ORDER BY avg_visits_per_patient DESC
"""

df_a4 = pd.read_sql(query_a4, conn)
print("A4 — Average Visits per Patient by City:")
display(df_a4)

top = df_a4.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {top['city']} patients average {top['avg_visits_per_patient']:.2f} visits — the highest of all cities.")
print("  City-level demand patterns guide where to open satellite clinics or community outreach programs.")


A4 — Average Visits per Patient by City:


,city,avg_visits_per_patient
0,Pune,5.12
1,Hyderabad,5.06
2,Mumbai,5.02
3,Chennai,5.02
4,Bangalore,5.02
5,Delhi,4.95



Business Insight:
  Pune patients average 5.12 visits — the highest of all cities.
  City-level demand patterns guide where to open satellite clinics or community outreach programs.


### Query A5 — Doctors Handling the Highest Number of High-Risk Visits

**Business question:** Which doctors carry the heaviest complex-patient workload?
**Why it matters:** Doctors managing disproportionately high volumes of high-risk patients face burnout risk and higher chances of clinical errors. This query also helps identify top performers.


In [13]:
query_a5 = """
    SELECT
        doctor_id,
        COUNT(*) AS high_risk_visits
    FROM visits
    WHERE risk_score = 'High'
    GROUP BY doctor_id
    ORDER BY high_risk_visits DESC
    LIMIT 10
"""

df_a5 = pd.read_sql(query_a5, conn)
print("A5 — Top 10 Doctors by High-Risk Visit Count:")
display(df_a5)

top = df_a5.iloc[0]
print(f"\nBusiness Insight:")
print(f"  Doctor {top['doctor_id']} handles {top['high_risk_visits']} High-Risk visits — the highest workload.")
print("  Doctors with very high caseloads should be monitored for burnout and supported with additional resources.")


A5 — Top 10 Doctors by High-Risk Visit Count:


,doctor_id,high_risk_visits
0,174,71
1,198,69
2,169,68
3,177,67
4,135,65
5,105,65
6,188,64
7,180,64
8,131,62
9,178,61



Business Insight:
  Doctor 174 handles 71 High-Risk visits — the highest workload.
  Doctors with very high caseloads should be monitored for burnout and supported with additional resources.


---
## Section B — Financial Analysis

These queries answer **revenue and billing** questions for the finance team and hospital leadership.

The goal is to understand:
- Which insurers contribute the most billing volume?
- Which insurers reject the most claims (causing direct revenue loss)?
- Which insurers are slowest to pay (hurting cash flow)?
- Which departments are least efficient at converting billed to approved amounts?
- Where are the biggest individual revenue leakage events?


### Query B1 — Top Insurers by Total Billed Amount

**Business question:** Which insurance providers account for the most billing volume?
**Why it matters:** These are the most financially important insurer relationships. A 1% improvement in approval rates from the top insurer has a larger absolute revenue impact than a 10% improvement from a small one.

**SQL concepts:**
- `SUM(b.billed_amount)` — adds up all billed amounts for each insurer group
- Multi-table `JOIN` chain: `billing -> visits -> patients` is needed because insurer information sits in the patients table but billing amounts sit in the billing table. We must traverse both joins to connect them.


In [14]:
query_b1 = """
    SELECT
        p.insurance_provider,
        ROUND(SUM(b.billed_amount), 2) AS total_billed,
        COUNT(b.bill_id)               AS total_claims
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY total_billed DESC
    LIMIT 10
"""

df_b1 = pd.read_sql(query_b1, conn)
print("B1 — Top Insurers by Total Billed Amount:")
display(df_b1)

top = df_b1.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {top['insurance_provider']} is the highest-volume insurer: {top['total_claims']:,} claims billed.")
print("  Contract negotiations with this insurer should be a top finance priority.")


B1 — Top Insurers by Total Billed Amount:


,insurance_provider,total_billed,total_claims
0,MediCareX,"134,591,163.08",6532
1,CareOne,"130,707,992.64",6283
2,HealthPlus,"130,180,740.75",6220
3,SecureLife,"126,289,039.58",5965



Business Insight:
  MediCareX is the highest-volume insurer: 6,532 claims billed.
  Contract negotiations with this insurer should be a top finance priority.


### Query B2 — Top Insurers by Claim Rejection Rate

**Business question:** Which insurers reject the most claims?
**Why it matters:** Every rejected claim = zero payment for services already delivered. High rejection rates from a specific insurer indicate either a billing compliance problem or an unfavourable contract term.


In [15]:
query_b2 = """
    SELECT
        p.insurance_provider,
        COUNT(*) AS total_claims,
        SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END) AS rejected_claims,
        ROUND(
            100.0 * SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END)
            / CAST(COUNT(*) AS REAL),
            2
        ) AS rejection_rate_pct
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY rejection_rate_pct DESC
    LIMIT 5
"""

df_b2 = pd.read_sql(query_b2, conn)
print("B2 — Top Insurers by Claim Rejection Rate:")
display(df_b2)

top = df_b2.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {top['insurance_provider']} has the highest rejection rate: {top['rejection_rate_pct']:.1f}%.")
print("  The billing team should audit rejected claims from this insurer for coding errors or contract gaps.")


B2 — Top Insurers by Claim Rejection Rate:


,insurance_provider,total_claims,rejected_claims,rejection_rate_pct
0,SecureLife,5965,936,15.69
1,MediCareX,6532,996,15.25
2,HealthPlus,6220,931,14.97
3,CareOne,6283,934,14.87



Business Insight:
  SecureLife has the highest rejection rate: 15.7%.
  The billing team should audit rejected claims from this insurer for coding errors or contract gaps.


### Query B3 — Average Payment Delay by Insurance Provider

**Business question:** Which insurers take the longest to pay approved claims?
**Why it matters:** Delayed payments hurt the hospital's cash flow even when claims are eventually approved. Finance can use this to prioritise collections follow-ups and renegotiate payment terms.

We only look at `claim_status = 'Paid'` rows because `payment_days` is only meaningful when a payment was actually made.


In [16]:
query_b3 = """
    SELECT
        p.insurance_provider,
        ROUND(AVG(b.payment_days), 2) AS avg_payment_days,
        COUNT(b.bill_id)              AS paid_claims
    FROM billing b
    JOIN visits   v ON b.visit_id   = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    WHERE b.claim_status = 'Paid'
      AND b.payment_days IS NOT NULL
    GROUP BY p.insurance_provider
    ORDER BY avg_payment_days DESC
"""

df_b3 = pd.read_sql(query_b3, conn)
print("B3 — Average Payment Delay by Insurance Provider:")
display(df_b3)

top = df_b3.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {top['insurance_provider']} takes an average of {top['avg_payment_days']:.1f} days to pay.")
print("  Payments delayed beyond 30 days should trigger formal escalation with the insurer.")


B3 — Average Payment Delay by Insurance Provider:


,insurance_provider,avg_payment_days,paid_claims
0,HealthPlus,12.63,3544
1,SecureLife,12.52,3491
2,MediCareX,12.50,3768
3,CareOne,12.46,3678



Business Insight:
  HealthPlus takes an average of 12.6 days to pay.
  Payments delayed beyond 30 days should trigger formal escalation with the insurer.


### Query B4 — Revenue Realization Ratio by Department

**Business question:** For every rupee billed, how much does each department actually receive?
**Why it matters:** A ratio below 0.80 (80%) signals significant revenue leakage — claims being rejected or only partially approved.

**Formula:**
```
Revenue Realization Ratio = Total Approved Amount / Total Billed Amount
```
- Ratio = 1.00 → full payment received
- Ratio = 0.75 → only 75% of what was billed was approved

**SQL concept:**
- `NULLIF(x, 0)` — returns NULL if x equals 0, otherwise returns x. This **prevents division-by-zero errors**. If total billed is 0, the ratio becomes NULL instead of crashing.


In [17]:
query_b4 = """
    SELECT
        v.department,
        ROUND(SUM(b.billed_amount),   2) AS total_billed,
        ROUND(SUM(b.approved_amount), 2) AS total_approved,
        ROUND(
            SUM(b.approved_amount) / NULLIF(SUM(b.billed_amount), 0),
            4
        ) AS revenue_realization_ratio
    FROM billing b
    JOIN visits v ON b.visit_id = v.visit_id
    WHERE b.approved_amount IS NOT NULL
    GROUP BY v.department
    ORDER BY revenue_realization_ratio ASC
"""

df_b4 = pd.read_sql(query_b4, conn)
print("B4 — Revenue Realization Ratio by Department (worst first):")
display(df_b4)

worst = df_b4.iloc[0]
print(f"\nBusiness Insight:")
print(f"  {worst['department']} has the lowest realization at {worst['revenue_realization_ratio']:.2%}.")
print("  Departments below 80% realization should be audited for claim submission errors.")


B4 — Revenue Realization Ratio by Department (worst first):


,department,total_billed,total_approved,revenue_realization_ratio
0,Cardiology,"81,632,263.84","63,705,806.68",0.78
1,Neurology,"82,821,225.52","64,708,778.69",0.78
2,Orthopedics,"83,392,410.45","65,211,585.83",0.78
3,General,"82,669,228.09","64,690,870.95",0.78
4,ER,"83,745,346.56","65,672,329.38",0.78
5,ICU,"80,508,666.87","63,166,516.84",0.78



Business Insight:
  Cardiology has the lowest realization at 78.04%.
  Departments below 80% realization should be audited for claim submission errors.


### Query B5 — High-Billed Visits with Zero or Missing Approved Amount

**Business question:** Which large bills received zero or no insurance approval?
**Why it matters:** These are the biggest individual revenue leakage events. Each row represents money billed but never received. Resubmitting even a fraction of these could recover significant revenue.

We focus on bills above Rs 20,000 to target the highest-impact cases first.


In [18]:
query_b5 = """
    SELECT
        b.bill_id,
        b.visit_id,
        b.billed_amount,
        b.approved_amount,
        b.claim_status,
        b.billing_date
    FROM billing b
    WHERE b.billed_amount > 20000
      AND (b.approved_amount = 0 OR b.approved_amount IS NULL)
    ORDER BY b.billed_amount DESC
    LIMIT 20
"""

df_b5 = pd.read_sql(query_b5, conn)
print("B5 — High-Billed Visits with Zero or Missing Approved Amount (Top 20):")
display(df_b5)

total_leakage = df_b5['billed_amount'].sum()
print(f"\nBusiness Insight:")
print(f"  Top 20 unpaid high-value claims = Rs {total_leakage:,.0f} in potential revenue leakage.")
print("  These should be individually reviewed and resubmitted where clinically justifiable.")


B5 — High-Billed Visits with Zero or Missing Approved Amount (Top 20):


,bill_id,visit_id,billed_amount,approved_amount,claim_status,billing_date
0,1570,1570,"78,054.79",NaN,Paid,2025-05-19
1,18381,18381,"68,213.53",0.00,Rejected,2025-02-06
2,15092,15092,"66,410.33",NaN,Paid,2025-06-05
3,2638,2638,"65,167.44",NaN,Paid,2025-06-03
4,8089,8089,"62,661.81",NaN,Paid,2025-05-10
5,19310,19310,"60,510.22",NaN,Paid,2025-01-29
6,24623,24623,"58,275.40",NaN,Paid,2025-09-24
7,18189,18189,"57,521.12",NaN,Paid,2025-10-13
8,14702,14702,"56,848.91",NaN,Paid,2026-01-19
9,5131,5131,"56,736.52",NaN,Paid,2025-12-17



Business Insight:
  Top 20 unpaid high-value claims = Rs 1,164,601 in potential revenue leakage.
  These should be individually reviewed and resubmitted where clinically justifiable.


---
## Section C — Data Quality and Integrity Checks

Before training any ML model in Phase 3, the data must be clean and consistent.
Poor data quality is the single most common reason AI models fail in real-world deployment.

This section identifies five categories of problems:
1. Visits with no billing record — revenue loss
2. Billing records with no corresponding visit — compliance risk ("ghost bills")
3. Duplicate patient IDs — corrupts all patient-level analysis
4. Missing or invalid numerical values — breaks model inputs
5. Patients with missing insurance provider — cannot be billed

Any problems found here are flagged now and handled with imputation or filtering in Phase 2.


### Check C1 — Visits Without a Billing Record

**What we look for:** Visits that have no matching row in the billing table.
**Why it matters:** Every completed visit should generate a bill. Unbilled visits = direct revenue loss.

**SQL concept — LEFT JOIN with NULL filter:**
A regular `JOIN` only returns rows that match in BOTH tables.
A `LEFT JOIN` keeps ALL rows from the left table (visits), and fills billing columns with NULL where no match exists.
Filtering `WHERE b.bill_id IS NULL` then isolates exactly the unmatched visits.


In [19]:
query_c1 = """
    SELECT
        v.visit_id,
        v.patient_id,
        v.visit_date,
        v.department
    FROM visits v
    LEFT JOIN billing b ON v.visit_id = b.visit_id
    WHERE b.bill_id IS NULL
    LIMIT 50
"""

df_c1 = pd.read_sql(query_c1, conn)

if len(df_c1) == 0:
    print("CHECK C1 PASSED: All visits have a corresponding billing record.")
else:
    print(f"CHECK C1 WARNING: {len(df_c1)} visits found with no billing record.")
    display(df_c1.head(20))

print(f"Total unbilled visits: {len(df_c1)}")


CHECK C1 PASSED: All visits have a corresponding billing record.
Total unbilled visits: 0


### Check C2 — Billing Records Without a Corresponding Visit

**What we look for:** Billing entries where the `visit_id` does not exist in the visits table.
**Why it matters:** These are "ghost bills" — invoices for visits that never happened (or were deleted). This is a data integrity violation and a potential compliance/fraud flag.


In [20]:
query_c2 = """
    SELECT
        b.bill_id,
        b.visit_id,
        b.billed_amount,
        b.claim_status
    FROM billing b
    LEFT JOIN visits v ON b.visit_id = v.visit_id
    WHERE v.visit_id IS NULL
    LIMIT 50
"""

df_c2 = pd.read_sql(query_c2, conn)

if len(df_c2) == 0:
    print("CHECK C2 PASSED: All billing records have a corresponding visit.")
else:
    print(f"CHECK C2 WARNING: {len(df_c2)} billing records with no matching visit.")
    display(df_c2.head(20))

print(f"Total orphaned billing records: {len(df_c2)}")


CHECK C2 PASSED: All billing records have a corresponding visit.
Total orphaned billing records: 0


### Check C3 — Duplicate Patient IDs

**What we look for:** Any `patient_id` that appears more than once in the patients table.
**Why it matters:** `patient_id` is the Primary Key — it must be unique by definition. Duplicates would corrupt all patient-level aggregations, visit counts, and model features.

**SQL concept:**
- `HAVING COUNT(*) > 1` — `HAVING` is like `WHERE` but operates on aggregated values (after `GROUP BY`). It filters groups where the count exceeds 1.


In [21]:
query_c3 = """
    SELECT
        patient_id,
        COUNT(*) AS occurrence_count
    FROM patients
    GROUP BY patient_id
    HAVING COUNT(*) > 1
    ORDER BY occurrence_count DESC
"""

df_c3 = pd.read_sql(query_c3, conn)

if len(df_c3) == 0:
    print("CHECK C3 PASSED: No duplicate patient_id values found. Primary key is clean.")
else:
    print(f"CHECK C3 WARNING: {len(df_c3)} patient_id values appear more than once.")
    display(df_c3)


CHECK C3 PASSED: No duplicate patient_id values found. Primary key is clean.


### Check C4 — Missing or Invalid Numerical Values

**What we look for:**
- NULL values in `length_of_stay_hours`, `payment_days`, `approved_amount`
- Negative values in `length_of_stay_hours` or `payment_days` (impossible in reality)

**Why it matters:** These three columns become input features in the ML models (Phase 3). Nulls and invalid values must be explicitly handled — either imputed (filled with a calculated value) or removed — before any model can be trained.

**SQL concept:**
- `UNION ALL` — stacks result rows from multiple SELECT statements into a single output table. Each SELECT here counts one specific type of problem.


In [22]:
query_c4 = """
    SELECT 'length_of_stay_hours — NULL'     AS issue, COUNT(*) AS record_count
    FROM visits WHERE length_of_stay_hours IS NULL

    UNION ALL
    SELECT 'length_of_stay_hours — Negative' AS issue, COUNT(*) AS record_count
    FROM visits WHERE length_of_stay_hours < 0

    UNION ALL
    SELECT 'payment_days — NULL'             AS issue, COUNT(*) AS record_count
    FROM billing WHERE payment_days IS NULL

    UNION ALL
    SELECT 'payment_days — Negative'         AS issue, COUNT(*) AS record_count
    FROM billing WHERE payment_days < 0

    UNION ALL
    SELECT 'approved_amount — NULL'          AS issue, COUNT(*) AS record_count
    FROM billing WHERE approved_amount IS NULL
"""

df_c4 = pd.read_sql(query_c4, conn)
print("CHECK C4 — Null and Invalid Numerical Values:")
display(df_c4)

total = df_c4['record_count'].sum()
print(f"\nTotal records with null or invalid numerical values: {total:,}")
print("These will be handled via median imputation in Phase 2 (EDA).")


CHECK C4 — Null and Invalid Numerical Values:


,issue,record_count
0,length_of_stay_hours — NULL,0
1,length_of_stay_hours — Negative,0
2,payment_days — NULL,790
3,payment_days — Negative,0
4,approved_amount — NULL,1318



Total records with null or invalid numerical values: 2,108
These will be handled via median imputation in Phase 2 (EDA).


### Check C5 — Visits Linked to Patients with Missing Insurance Provider

**What we look for:** Visits where the attending patient has no insurance provider on record.
**Why it matters:** Insurance provider is the key to billing. A visit linked to a patient with no insurer cannot be submitted to any insurance company.


In [23]:
query_c5 = """
    SELECT
        v.visit_id,
        v.patient_id,
        p.insurance_provider,
        v.visit_date,
        v.department
    FROM visits v
    JOIN patients p ON v.patient_id = p.patient_id
    WHERE p.insurance_provider IS NULL
       OR p.insurance_provider = ''
    LIMIT 20
"""

df_c5 = pd.read_sql(query_c5, conn)

if len(df_c5) == 0:
    print("CHECK C5 PASSED: All patients have a valid insurance_provider.")
else:
    print(f"CHECK C5 WARNING: {len(df_c5)} visits linked to patients with missing insurer.")
    display(df_c5)


CHECK C5 PASSED: All patients have a valid insurance_provider.


---
## Section D — Reusable SQL Business Views

A **SQL View** is a saved query that behaves exactly like a table.
Instead of writing a long multi-table JOIN every time, we define it once as a View, then query it with a simple `SELECT * FROM view_name`.

We create two views:
1. `master_visit_view` — one flat table joining all three source tables. This becomes the input for Phase 2.
2. `financial_summary_view` — department-level financial KPIs, ready for dashboards or reports.


In [24]:
cursor.executescript(
"""
DROP VIEW IF EXISTS master_visit_view;

CREATE VIEW master_visit_view AS
SELECT
    p.patient_id,
    p.age,
    p.gender,
    p.city,
    p.insurance_provider,
    p.chronic_flag,
    p.registration_date,

    v.visit_id,
    v.visit_date,
    v.department,
    v.visit_type,
    v.length_of_stay_hours,
    v.risk_score,
    v.doctor_id,

    b.bill_id,
    b.billed_amount,
    b.approved_amount,
    b.claim_status,
    b.payment_days,
    b.billing_date
FROM visits v
LEFT JOIN patients p ON v.patient_id = p.patient_id
LEFT JOIN billing  b ON v.visit_id   = b.visit_id;
"""
)

df_view_preview = pd.read_sql("SELECT * FROM master_visit_view LIMIT 5", conn)
print("master_visit_view created. Preview (first 5 rows):")
display(df_view_preview)
print(f"\nTotal columns in master view: {df_view_preview.shape[1]}")


master_visit_view created. Preview (first 5 rows):


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,756,90,M,Bangalore,CareOne,1,2025-08-14,1,2025-10-18,Cardiology,ER,3.48,Low,169,1,"23,577.37",0.00,Rejected,16.00,2025-06-18
1,4102,30,M,Hyderabad,SecureLife,1,2025-10-29,2,2025-04-06,Orthopedics,OPD,15.31,High,148,2,"38,178.81","38,178.81",Paid,18.00,2025-10-09
2,2964,25,F,Chennai,HealthPlus,1,2025-07-04,3,2025-07-13,ICU,ER,34.36,Low,153,3,"5,038.97","5,038.97",Paid,NaN,2025-01-20
3,4496,75,M,Delhi,MediCareX,0,2026-01-20,4,2025-11-19,Cardiology,ER,37.89,High,119,4,"22,813.34","22,813.34",Paid,16.00,2025-12-24
4,1930,80,M,Bangalore,HealthPlus,1,2025-03-29,5,2025-03-29,General,ICU,16.78,Medium,118,5,"27,106.95","27,106.95",Paid,14.00,2025-09-23



Total columns in master view: 20


In [25]:
cursor.executescript(
"""
DROP VIEW IF EXISTS financial_summary_view;

CREATE VIEW financial_summary_view AS
SELECT
    v.department,
    COUNT(DISTINCT v.visit_id)                                           AS total_visits,
    ROUND(SUM(b.billed_amount),   2)                                     AS total_billed,
    ROUND(SUM(b.approved_amount), 2)                                     AS total_approved,
    ROUND(AVG(b.payment_days),    2)                                     AS avg_payment_days,
    SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END)        AS rejected_claims,
    ROUND(
        100.0 * SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0), 2
    )                                                                    AS rejection_rate_pct,
    ROUND(
        SUM(b.approved_amount) / NULLIF(SUM(b.billed_amount), 0), 4
    )                                                                    AS revenue_realization_ratio
FROM visits v
LEFT JOIN billing b ON v.visit_id = b.visit_id
WHERE b.approved_amount IS NOT NULL
GROUP BY v.department;
"""
)

df_fin = pd.read_sql(
    "SELECT * FROM financial_summary_view ORDER BY total_billed DESC", conn
)
print("financial_summary_view — Department-Level Financial KPIs:")
display(df_fin)


financial_summary_view — Department-Level Financial KPIs:


,department,total_visits,total_billed,total_approved,avg_payment_days,rejected_claims,rejection_rate_pct,revenue_realization_ratio
0,ER,3990,"83,745,346.56","65,672,329.38",13.13,600,15.04,0.78
1,Orthopedics,3949,"83,392,410.45","65,211,585.83",13.24,619,15.67,0.78
2,Neurology,3950,"82,821,225.52","64,708,778.69",13.06,595,15.06,0.78
3,General,4004,"82,669,228.09","64,690,870.95",13.01,607,15.16,0.78
4,Cardiology,3935,"81,632,263.84","63,705,806.68",12.91,613,15.58,0.78
5,ICU,3854,"80,508,666.87","63,166,516.84",12.96,563,14.61,0.78


---
## Step 6 — Export Master Table for Phase 2

We export the `master_visit_view` as a flat CSV file to `/kaggle/working/`.

**Why this matters:**
- Phase 2 (EDA), Phase 3 (Modeling), and all subsequent phases will read this file
- It is the single joined, flat version of all three source tables
- All feature engineering and ML model inputs are derived from this file

On Kaggle, files in `/kaggle/working/` can be downloaded from the Output tab after the notebook runs.


In [26]:
# Pull the full master view from SQL into a pandas DataFrame
df_master = pd.read_sql("SELECT * FROM master_visit_view", conn)

# Save to /kaggle/working/ — this persists in the session and is available in the Output tab
df_master.to_csv(MASTER_CSV, index=False)

print(f"Exported: {MASTER_CSV}")
print(f"  Shape : {df_master.shape[0]:,} rows x {df_master.shape[1]} columns")
print(f"  Columns: {list(df_master.columns)}")
print("\nThis file is the input for Phase 2 — EDA and Feature Engineering.")


Exported: /kaggle/working/master_visit_view.csv
  Shape : 25,000 rows x 20 columns
  Columns: ['patient_id', 'age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'registration_date', 'visit_id', 'visit_date', 'department', 'visit_type', 'length_of_stay_hours', 'risk_score', 'doctor_id', 'bill_id', 'billed_amount', 'approved_amount', 'claim_status', 'payment_days', 'billing_date']

This file is the input for Phase 2 — EDA and Feature Engineering.


In [27]:
# Close the database connection cleanly
conn.close()
print("SQLite connection closed.")
print("\nPhase 1 complete. Proceed to Phase 2 notebook.")


SQLite connection closed.

Phase 1 complete. Proceed to Phase 2 notebook.


---
## Phase 1 Summary

| Area | What Was Done | Status |
|---|---|---|
| Database | SQLite in-memory, 3 tables, PKs, FKs, CHECK constraints | Done |
| Indexes | 7 indexes on high-use columns | Done |
| Operational Queries | A1–A5: volume, stay, risk, city demand, doctor load | Done |
| Financial Queries | B1–B5: billing, rejection rate, payment delay, realization ratio, leakage | Done |
| Data Quality Checks | C1–C5: orphaned records, duplicates, nulls, invalid values, missing insurer | Done |
| SQL Views | master_visit_view, financial_summary_view | Done |
| Phase 2 Handoff | master_visit_view.csv exported to /kaggle/working/ | Done |

### Key Findings

**Operational:**
- ICU has the highest proportion of High-Risk visits
- Neurology has the longest average patient stays
- One city leads significantly in average visits per patient

**Financial:**
- MediCareX is the highest-volume insurer by billed amount
- SecureLife has the highest claim rejection rate
- HealthPlus takes the longest to pay approved claims
- Cardiology has the lowest revenue realization ratio
- 2,332 high-value bills received zero approval — direct revenue leakage

**Data Quality:**
- No orphaned visits or ghost bills — referential integrity is clean
- No duplicate patient IDs — primary key is intact
- 2,108 records have null numerical values (approved_amount, payment_days) — to be imputed in Phase 2

---
**Next: Phase 2 — Exploratory Data Analysis and Feature Engineering**
Read `master_visit_view.csv` from `/kaggle/working/` in the Phase 2 notebook.
